Github Analysis!

In [1]:
# Use the correct function name from your file
from spark_utils import load_github_bronze_keyword_extractions_data
spark, df = load_github_bronze_keyword_extractions_data()

🚀 Creating Spark session: GitHubAnalysis
📡 Connecting to: spark://spark-master:7077
🗄️  MinIO endpoint: http://minio:9000
🔺 Delta Lake: ENABLED


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/24 05:24:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/06/24 05:24:03 WARN Utils: Service 'sparkDriver' could not bind on port 4040. Attempting port 4041.
25/06/24 05:24:03 WARN Utils: Service 'sparkDriver' could not bind on port 4041. Attempting port 4042.
25/06/24 05:24:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/06/24 05:24:04 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/06/24 05:24:04 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


✅ Spark 4.0.0 session created successfully!
🔗 Spark UI: http://localhost:4041
💡 S3A ready for s3a://delta-lake/ operations
🔺 Delta Lake ready for delta table operations!


25/06/24 05:24:35 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


AnalysisException: [PATH_NOT_FOUND] Path does not exist: s3a://delta-lake/bronze/github/keyword_extractions. SQLSTATE: 42K03

In [ ]:
# Simpler version without imports
df = spark.read.format("delta").load("s3a://delta-lake/bronze/github/keyword_extractions")

print("=== TOP KEYWORDS (Simple) ===")
df.groupBy("keyword") \
  .sum("mentions") \
  .orderBy("sum(mentions)", ascending=False) \
  .limit(10) \
  .show(truncate=False)

print("\n=== BASIC STATS ===")
df.describe("mentions").show()
df.select("keyword").distinct().count()

In [5]:
# Load your data
df = spark.read.format("delta").load("s3a://delta-lake/bronze/github/keyword_extractions")

print("=== AI/ML TRENDS IN GITHUB ACTIVITY ===")
# Filter for AI-related keywords from your taxonomy
ai_keywords = ['gpt-4o', 'claude', 'llama', 'pytorch', 'tensorflow', 'transformers', 'langchain']

ai_df = df.filter(df.keyword.isin(ai_keywords))
ai_df.groupBy("keyword") \
  .sum("mentions") \
  .orderBy("sum(mentions)", ascending=False) \
  .show()

print("\n=== BLOCKCHAIN ACTIVITY ===")
blockchain_keywords = ['ethereum', 'solana', 'near', 'web3', 'solidity']
blockchain_df = df.filter(df.keyword.isin(blockchain_keywords))
blockchain_df.groupBy("keyword") \
  .sum("mentions") \
  .orderBy("sum(mentions)", ascending=False) \
  .show()

print("\n=== DEVOPS TRENDS ===")
devops_keywords = ['docker', 'kubernetes', 'terraform', 'github actions', 'jenkins']
devops_df = df.filter(df.keyword.isin(devops_keywords))
devops_df.groupBy("keyword") \
  .sum("mentions") \
  .orderBy("sum(mentions)", ascending=False) \
  .show()

print("\n=== HOURLY DEVELOPER ACTIVITY PATTERN ===")
df.groupBy("hour") \
  .sum("mentions") \
  .orderBy("hour") \
  .show()

=== AI/ML TRENDS IN GITHUB ACTIVITY ===
+------------+-------------+
|     keyword|sum(mentions)|
+------------+-------------+
|       llama|          514|
|     pytorch|          452|
|  tensorflow|          303|
|   langchain|          284|
|transformers|          221|
|      claude|          220|
|      gpt-4o|           65|
+------------+-------------+


=== BLOCKCHAIN ACTIVITY ===
+--------+-------------+
| keyword|sum(mentions)|
+--------+-------------+
|ethereum|        31544|
|    near|        31156|
|  solana|          196|
|    web3|           90|
|solidity|           83|
+--------+-------------+


=== DEVOPS TRENDS ===
+--------------+-------------+
|       keyword|sum(mentions)|
+--------------+-------------+
|        docker|        14178|
|github actions|         9389|
|    kubernetes|         1832|
|     terraform|         1699|
|       jenkins|         1190|
+--------------+-------------+


=== HOURLY DEVELOPER ACTIVITY PATTERN ===
+----+-------------+
|hour|sum(mentions

In [6]:
# Let's see the complete hourly pattern and analyze peak times
print("=== COMPLETE HOURLY PATTERN ===")
hourly_pattern = df.groupBy("hour") \
  .sum("mentions") \
  .orderBy("hour") \
  .collect()

for row in hourly_pattern:
    hour = row['hour']
    mentions = row['sum(mentions)']
    # Convert to visual bar
    bar = "█" * max(1, int(mentions / 2000))
    print(f"Hour {hour:2d}: {mentions:6,} {bar}")

print("\n=== NEAR vs ETHEREUM DEEP DIVE ===")
# Why is NEAR so close to Ethereum?
near_vs_eth = df.filter(df.keyword.isin(['near', 'ethereum']))
near_vs_eth.groupBy("keyword", "hour") \
  .sum("mentions") \
  .orderBy("keyword", "hour") \
  .show(50)

=== COMPLETE HOURLY PATTERN ===
Hour  0: 50,714 █████████████████████████
Hour  1: 56,864 ████████████████████████████
Hour  2: 31,619 ███████████████
Hour  3: 25,930 ████████████
Hour  4: 26,269 █████████████
Hour  5: 26,620 █████████████
Hour  6: 32,782 ████████████████
Hour  7: 27,381 █████████████
Hour  8: 27,906 █████████████
Hour  9: 28,442 ██████████████
Hour 10: 30,076 ███████████████
Hour 11: 27,262 █████████████
Hour 12: 75,622 █████████████████████████████████████
Hour 13: 49,523 ████████████████████████
Hour 14: 28,816 ██████████████
Hour 15: 31,231 ███████████████
Hour 16: 30,267 ███████████████
Hour 17: 27,067 █████████████
Hour 18: 42,152 █████████████████████
Hour 19: 25,155 ████████████
Hour 20: 28,604 ██████████████
Hour 21: 23,954 ███████████
Hour 22: 22,795 ███████████
Hour 23: 21,320 ██████████

=== NEAR vs ETHEREUM DEEP DIVE ===
+--------+----+-------------+
| keyword|hour|sum(mentions)|
+--------+----+-------------+
|ethereum|   0|          193|
|ethereum|   1|  